# 06 — LlamaIndex Integration: Filtered Hybrid Retrieval with SimlarVectorStore

`SimlarVectorStore` is a LlamaIndex vector store backed by simlar's `FilteredIndex` over a `HelixIndex`.
It plugs into the standard `VectorStoreIndex` pipeline, so retrievers, query engines and chat engines work unchanged.

**What we cover**
- Indexing documents with metadata through `VectorStoreIndex`
- Query modes: `default` (vector), `sparse` (BM25) and `hybrid` (BM25 + vector, RRF-fused)
- Metadata filtering with LlamaIndex `MetadataFilters`, applied *before* ranking
- Role-based access with `vector_store_kwargs={"roles": [...]}`
- Deleting documents and persisting with `StorageContext`
- Wiring into a query engine

In [ ]:
%pip install -q datasets llama-index-core llama-index-embeddings-huggingface simlar

## Load the dataset

We use [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — 120,000 news headlines across four categories.
We index 500 articles, keeping the category as document metadata so we can filter on it.

In [ ]:
from datasets import load_dataset
from llama_index.core import Document

LABEL_NAMES = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

ds = load_dataset("fancyzhx/ag_news", split="train[:500]")
documents = [
    Document(text=text, id_=str(i), metadata={"category": LABEL_NAMES[label], "length": len(text)})
    for i, (text, label) in enumerate(zip(ds["text"], ds["label"]))
]

print(f"Loaded {len(documents)} articles")
print(f"Sample: {documents[0].text[:120]}  {documents[0].metadata}")

## Build the index

`SimlarVectorStore` is embedding-agnostic — LlamaIndex embeds the documents with whatever `embed_model` you pass.
Here we use `HuggingFaceEmbedding` with `BAAI/bge-small-en-v1.5` (384-dim, runs on CPU).

Every write goes straight to the index: adding more documents later appends only those documents.

In [ ]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from simlar.integrations.llama_index.simlar_vector_store import SimlarVectorStore

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

vector_store = SimlarVectorStore()
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents[:300], storage_context=storage_context, embed_model=embed_model, show_progress=True
)
for doc in documents[300:]:  # incremental inserts
    index.insert(doc)

print(f"Indexed nodes: {len(vector_store.client.ids)}")

## Query modes

`vector_store_query_mode` picks the search, following the same convention as `PGVectorStore`:

| Mode | Search |
|---|---|
| `"default"` | vector similarity |
| `"sparse"` | BM25 text search |
| `"hybrid"` | BM25 + vector, fused with RRF |

In [ ]:
def show(nodes):
    for n in nodes:
        print(f"  [{n.node.metadata['category']:8s}] score={n.score:.4f}  {n.node.get_content()[:90]}")
    print()


question = "interest rate Federal Reserve inflation"
for mode in ("default", "sparse", "hybrid"):
    print(f"mode={mode}")
    show(index.as_retriever(similarity_top_k=3, vector_store_query_mode=mode).retrieve(question))

## Metadata filtering

Pass LlamaIndex `MetadataFilters` to the retriever. They are translated to simlar's SQL metadata filter and resolved to the
allowed documents *before* ranking, so the results are the exact top-k among them.

Supported operators: `==`, `!=`, `>`, `>=`, `<`, `<=`, `in`, `nin`, `contains`, `any`, `all`, `text_match`,
`text_match_insensitive`, `is_empty`; conditions `and`, `or`, `not`, nested to any depth. A key no document has matches nothing.

In [ ]:
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters

filters = MetadataFilters(
    filters=[
        MetadataFilter(key="category", value=["Business", "Sci/Tech"], operator="in"),
        MetadataFilter(key="length", value=200, operator=">="),
    ],
    condition="and",
)
retriever = index.as_retriever(similarity_top_k=5, vector_store_query_mode="hybrid", filters=filters)
show(retriever.retrieve("technology startup funding Silicon Valley"))

## Role-based access

Grant roles when adding nodes (`vector_store.add(nodes, roles=[...])`) or afterwards with `grant` / `revoke`.
Queries pass `roles` through `vector_store_kwargs`; only nodes granted to one of those roles are returned.

In [ ]:
sports_ids = [n.node_id for n in vector_store.get_nodes(filters=MetadataFilters(filters=[MetadataFilter(key="category", value="Sports")]))]
vector_store.grant(sports_ids, ["sports_desk"])

retriever = index.as_retriever(similarity_top_k=3, vector_store_kwargs={"roles": ["sports_desk"]})
show(retriever.retrieve("record breaking performance"))

## Delete and persist

`index.delete_ref_doc(doc_id)` removes a document's nodes. `StorageContext.persist()` saves the store;
`SimlarVectorStore.from_persist_dir()` loads it back without re-embedding.

In [ ]:
import tempfile

index.delete_ref_doc("0")

with tempfile.TemporaryDirectory() as tmp:
    index.storage_context.persist(persist_dir=tmp)

    reloaded_store = SimlarVectorStore.from_persist_dir(tmp)
    reloaded = VectorStoreIndex.from_vector_store(reloaded_store, embed_model=embed_model)

    q = "Federal Reserve interest rate decision"
    original = [n.node.node_id for n in index.as_retriever(similarity_top_k=5).retrieve(q)]
    restored = [n.node.node_id for n in reloaded.as_retriever(similarity_top_k=5).retrieve(q)]
    print(f"Results match after reload: {original == restored}")

## Wire into a query engine

> **Note** — `query()` calls an LLM to synthesize a response. Configure `llama_index.core.Settings.llm` first
> (e.g. `%pip install -q llama-index-llms-openai`). Retrieval itself works without an LLM.

In [ ]:
# from llama_index.llms.openai import OpenAI
# from llama_index.core import Settings
# Settings.llm = OpenAI(model="gpt-4o-mini")

query_engine = index.as_query_engine(
    similarity_top_k=5,
    vector_store_query_mode="hybrid",
    filters=MetadataFilters(filters=[MetadataFilter(key="category", value="Sci/Tech")]),
)
# print(query_engine.query("What are recent developments in space technology?"))